# Lab 12: Simulated Data, PCA, and K-means Clustering

**AAE722 Machine Learning**  
**Kingsley Hantian Ye**

This lab explores:
- Data simulation with distinct classes
- Principal Component Analysis (PCA)
- K-means clustering with different parameters
- Comparison of clustering results with true labels


## Import Libraries


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12


## Part (a): Generate a Simulated Data Set

**Requirements:**
- 20 observations for each of three classes (total: 60 observations)
- 50 variables (features)
- Ensure there's a "mean shift" between observations of each class so that the three classes are distinct


In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Parameters
n_classes = 3
n_obs_per_class = 20
n_total = n_classes * n_obs_per_class  # 60 observations
n_features = 50  # 50 variables

# Generate data with mean shifts for each class
# Class 0: mean around 0
# Class 1: mean shift of +3
# Class 2: mean shift of -3
mean_shifts = [0, 3, -3]

# Generate data
data_list = []
true_labels = []

for class_idx in range(n_classes):
    # Generate 20 observations for this class
    # Each observation has 50 features
    # Add mean shift to make classes distinct
    class_data = np.random.normal(
        loc=mean_shifts[class_idx],  # Mean shift
        scale=1.0,  # Standard deviation
        size=(n_obs_per_class, n_features)
    )
    data_list.append(class_data)
    true_labels.extend([class_idx] * n_obs_per_class)

# Combine all data
X = np.vstack(data_list)
true_labels = np.array(true_labels)

# Create DataFrame for easier manipulation
feature_names = [f'X{i+1}' for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)
df['True_Class'] = true_labels

print("="*70)
print("DATA GENERATION:")
print("="*70)
print(f"\nTotal observations: {n_total}")
print(f"Number of features: {n_features}")
print(f"Number of classes: {n_classes}")
print(f"Observations per class: {n_obs_per_class}")

print(f"\nTrue class distribution:")
print(pd.Series(true_labels).value_counts().sort_index())

print(f"\nMean values for each class (first 5 features):")
for class_idx in range(n_classes):
    class_data = X[true_labels == class_idx]
    print(f"  Class {class_idx}: {class_data[:, :5].mean(axis=0)}")

print(f"\nData shape: {X.shape}")
print(f"First 5 rows (first 5 features):")
print(df[feature_names[:5]].head())


## Part (b): Perform PCA and Plot Results

**Task:**
- Apply PCA to the 60 observations
- Plot the first two principal component score vectors
- Use different colors for observations belonging to each of the three true classes
- If classes are not separated, return to part (a) and modify the data simulation


In [ ]:
# Perform PCA
pca = PCA()
X_pca = pca.fit_transform(X)

# Get first two principal components
PC1 = X_pca[:, 0]
PC2 = X_pca[:, 1]

# Calculate variance explained
variance_explained = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_explained)

print("="*70)
print("PCA RESULTS:")
print("="*70)
print(f"\nVariance explained by first PC: {variance_explained[0]:.4f} ({variance_explained[0]*100:.2f}%)")
print(f"Variance explained by second PC: {variance_explained[1]:.4f} ({variance_explained[1]*100:.2f}%)")
print(f"Total variance explained by first 2 PCs: {cumulative_variance[1]:.4f} ({cumulative_variance[1]*100:.2f}%)")

# Plot PCA results
fig, ax = plt.subplots(figsize=(10, 8))

# Plot each class with different colors
colors = ['blue', 'red', 'green']
class_names = ['Class 0', 'Class 1', 'Class 2']

for class_idx in range(n_classes):
    mask = true_labels == class_idx
    ax.scatter(PC1[mask], PC2[mask], 
               c=colors[class_idx], s=100, alpha=0.6, 
               label=class_names[class_idx], 
               edgecolors='black', linewidths=1)

ax.set_xlabel('First Principal Component', fontsize=14)
ax.set_ylabel('Second Principal Component', fontsize=14)
ax.set_title('PCA: First Two Principal Components', fontsize=16, fontweight='bold')
ax.legend(loc='best', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("CLASS SEPARATION CHECK:")
print("="*70)

# Check if classes are separated
# Calculate centroids for each class in PC space
centroids = []
for class_idx in range(n_classes):
    mask = true_labels == class_idx
    centroid = np.array([PC1[mask].mean(), PC2[mask].mean()])
    centroids.append(centroid)
    print(f"\nClass {class_idx} centroid in PC space: ({centroid[0]:.3f}, {centroid[1]:.3f})")

# Calculate distances between centroids
distances = []
for i in range(n_classes):
    for j in range(i+1, n_classes):
        dist = np.linalg.norm(centroids[i] - centroids[j])
        distances.append(dist)
        print(f"Distance between Class {i} and Class {j}: {dist:.3f}")

min_distance = min(distances)
print(f"\nMinimum distance between class centroids: {min_distance:.3f}")

if min_distance > 2.0:
    print("✓ Classes appear to be well separated in the first two principal components.")
    print("  Proceeding to part (c)...")
else:
    print("⚠ Classes may not be well separated. Consider adjusting mean shifts in part (a).")
    print("  However, proceeding with current data...")


In [ ]:
# Perform K-means clustering with K=3
kmeans3 = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels_3 = kmeans3.fit_predict(X)

# Create crosstab to compare true labels with cluster labels
crosstab_3 = pd.crosstab(pd.Series(true_labels, name='True Class'),
                         pd.Series(cluster_labels_3, name='Cluster'),
                         margins=True)

print("="*70)
print("K-MEANS CLUSTERING (K=3):")
print("="*70)
print("\nCrosstab: True Class vs Cluster Assignment")
print(crosstab_3)

# Calculate accuracy (need to map cluster labels to true labels)
# Since K-means assigns arbitrary numbers, we need to find the best mapping
# Create cost matrix for Hungarian algorithm
cost_matrix = np.zeros((3, 3))
for true_class in range(3):
    for cluster in range(3):
        # Count how many observations of true_class are in cluster
        count = np.sum((true_labels == true_class) & (cluster_labels_3 == cluster))
        # Use negative count as cost (we want to maximize matches)
        cost_matrix[true_class, cluster] = -count

# Find optimal assignment
row_ind, col_ind = linear_sum_assignment(cost_matrix)

# Calculate accuracy with optimal mapping
correct = 0
for true_class, cluster in zip(row_ind, col_ind):
    correct += np.sum((true_labels == true_class) & (cluster_labels_3 == cluster))

accuracy_3 = correct / len(true_labels)

print(f"\nOptimal cluster-to-class mapping:")
for true_class, cluster in zip(row_ind, col_ind):
    print(f"  True Class {true_class} → Cluster {cluster}")

print(f"\nAccuracy (with optimal mapping): {accuracy_3:.4f} ({accuracy_3*100:.2f}%)")

# Visualize clustering results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: True labels
for class_idx in range(n_classes):
    mask = true_labels == class_idx
    ax1.scatter(PC1[mask], PC2[mask], 
               c=colors[class_idx], s=100, alpha=0.6, 
               label=class_names[class_idx], 
               edgecolors='black', linewidths=1)
ax1.set_xlabel('First Principal Component', fontsize=12)
ax1.set_ylabel('Second Principal Component', fontsize=12)
ax1.set_title('True Class Labels', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Cluster assignments
for cluster_idx in range(3):
    mask = cluster_labels_3 == cluster_idx
    ax2.scatter(PC1[mask], PC2[mask], 
               c=colors[cluster_idx], s=100, alpha=0.6, 
               label=f'Cluster {cluster_idx}', 
               edgecolors='black', linewidths=1)
ax2.set_xlabel('First Principal Component', fontsize=12)
ax2.set_ylabel('Second Principal Component', fontsize=12)
ax2.set_title('K-means Clustering (K=3)', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Part (d): Perform K-means Clustering (K=2)

**Task:** Perform K-means clustering with K = 2 and describe the results.


In [ ]:
# Perform K-means clustering with K=2
kmeans2 = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels_2 = kmeans2.fit_predict(X)

# Create crosstab
crosstab_2 = pd.crosstab(pd.Series(true_labels, name='True Class'),
                         pd.Series(cluster_labels_2, name='Cluster'),
                         margins=True)

print("="*70)
print("K-MEANS CLUSTERING (K=2):")
print("="*70)
print("\nCrosstab: True Class vs Cluster Assignment")
print(crosstab_2)

# Analyze which classes are grouped together
print("\n" + "="*70)
print("ANALYSIS:")
print("="*70)
for cluster in range(2):
    mask = cluster_labels_2 == cluster
    class_distribution = pd.Series(true_labels[mask]).value_counts().sort_index()
    print(f"\nCluster {cluster} contains:")
    for true_class, count in class_distribution.items():
        print(f"  - {count} observations from True Class {true_class} ({count/n_obs_per_class*100:.1f}% of class)")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: True labels
for class_idx in range(n_classes):
    mask = true_labels == class_idx
    ax1.scatter(PC1[mask], PC2[mask], 
               c=colors[class_idx], s=100, alpha=0.6, 
               label=class_names[class_idx], 
               edgecolors='black', linewidths=1)
ax1.set_xlabel('First Principal Component', fontsize=12)
ax1.set_ylabel('Second Principal Component', fontsize=12)
ax1.set_title('True Class Labels', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Cluster assignments (K=2)
cluster_colors_2 = ['purple', 'orange']
for cluster_idx in range(2):
    mask = cluster_labels_2 == cluster_idx
    ax2.scatter(PC1[mask], PC2[mask], 
               c=cluster_colors_2[cluster_idx], s=100, alpha=0.6, 
               label=f'Cluster {cluster_idx}', 
               edgecolors='black', linewidths=1)
ax2.set_xlabel('First Principal Component', fontsize=12)
ax2.set_ylabel('Second Principal Component', fontsize=12)
ax2.set_title('K-means Clustering (K=2)', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DESCRIPTION:")
print("="*70)
print("With K=2, K-means clustering forces the data into 2 clusters.")
print("This means that two of the three true classes will be grouped together,")
print("while the third class may be split or grouped with one of the others.")
print("The algorithm chooses the partition that minimizes within-cluster variance.")


## Part (e): Perform K-means Clustering (K=4)

**Task:** Perform K-means clustering with K = 4 and describe the results.


In [ ]:
# Perform K-means clustering with K=4
kmeans4 = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels_4 = kmeans4.fit_predict(X)

# Create crosstab
crosstab_4 = pd.crosstab(pd.Series(true_labels, name='True Class'),
                         pd.Series(cluster_labels_4, name='Cluster'),
                         margins=True)

print("="*70)
print("K-MEANS CLUSTERING (K=4):")
print("="*70)
print("\nCrosstab: True Class vs Cluster Assignment")
print(crosstab_4)

# Analyze which classes are split
print("\n" + "="*70)
print("ANALYSIS:")
print("="*70)
for cluster in range(4):
    mask = cluster_labels_4 == cluster
    class_distribution = pd.Series(true_labels[mask]).value_counts().sort_index()
    print(f"\nCluster {cluster} contains:")
    for true_class, count in class_distribution.items():
        if count > 0:
            print(f"  - {count} observations from True Class {true_class} ({count/n_obs_per_class*100:.1f}% of class)")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: True labels
for class_idx in range(n_classes):
    mask = true_labels == class_idx
    ax1.scatter(PC1[mask], PC2[mask], 
               c=colors[class_idx], s=100, alpha=0.6, 
               label=class_names[class_idx], 
               edgecolors='black', linewidths=1)
ax1.set_xlabel('First Principal Component', fontsize=12)
ax1.set_ylabel('Second Principal Component', fontsize=12)
ax1.set_title('True Class Labels', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Cluster assignments (K=4)
cluster_colors_4 = ['purple', 'orange', 'brown', 'pink']
for cluster_idx in range(4):
    mask = cluster_labels_4 == cluster_idx
    ax2.scatter(PC1[mask], PC2[mask], 
               c=cluster_colors_4[cluster_idx], s=100, alpha=0.6, 
               label=f'Cluster {cluster_idx}', 
               edgecolors='black', linewidths=1)
ax2.set_xlabel('First Principal Component', fontsize=12)
ax2.set_ylabel('Second Principal Component', fontsize=12)
ax2.set_title('K-means Clustering (K=4)', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DESCRIPTION:")
print("="*70)
print("With K=4, K-means clustering creates 4 clusters.")
print("Since there are only 3 true classes, at least one true class will be split")
print("into multiple clusters, or one cluster will contain observations from")
print("multiple true classes. The algorithm partitions the data to minimize")
print("within-cluster variance, which may not align with the true class structure.")


## Part (f): Perform K-means Clustering on PCA Scores (K=3)

**Task:**
- Perform K-means clustering with K = 3
- Use the **first two principal component score vectors** as input (60 x 2 matrix)
- Comment on the results


In [ ]:
# Use first two principal components as input
X_pca_2d = X_pca[:, :2]  # 60 x 2 matrix

# Perform K-means clustering on PCA scores
kmeans3_pca = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels_3_pca = kmeans3_pca.fit_predict(X_pca_2d)

# Create crosstab
crosstab_3_pca = pd.crosstab(pd.Series(true_labels, name='True Class'),
                              pd.Series(cluster_labels_3_pca, name='Cluster'),
                              margins=True)

print("="*70)
print("K-MEANS CLUSTERING ON PCA SCORES (K=3):")
print("="*70)
print("\nInput: First two principal component score vectors (60 x 2 matrix)")
print("\nCrosstab: True Class vs Cluster Assignment")
print(crosstab_3_pca)

# Calculate accuracy
cost_matrix_pca = np.zeros((3, 3))
for true_class in range(3):
    for cluster in range(3):
        count = np.sum((true_labels == true_class) & (cluster_labels_3_pca == cluster))
        cost_matrix_pca[true_class, cluster] = -count

row_ind_pca, col_ind_pca = linear_sum_assignment(cost_matrix_pca)

correct_pca = 0
for true_class, cluster in zip(row_ind_pca, col_ind_pca):
    correct_pca += np.sum((true_labels == true_class) & (cluster_labels_3_pca == cluster))

accuracy_3_pca = correct_pca / len(true_labels)

print(f"\nOptimal cluster-to-class mapping:")
for true_class, cluster in zip(row_ind_pca, col_ind_pca):
    print(f"  True Class {true_class} → Cluster {cluster}")

print(f"\nAccuracy (with optimal mapping): {accuracy_3_pca:.4f} ({accuracy_3_pca*100:.2f}%)")

# Compare with part (c)
print(f"\nComparison with clustering on original data (Part c):")
print(f"  Accuracy on original data: {accuracy_3:.4f} ({accuracy_3*100:.2f}%)")
print(f"  Accuracy on PCA scores:   {accuracy_3_pca:.4f} ({accuracy_3_pca*100:.2f}%)")
print(f"  Difference: {abs(accuracy_3 - accuracy_3_pca):.4f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: True labels
for class_idx in range(n_classes):
    mask = true_labels == class_idx
    ax1.scatter(PC1[mask], PC2[mask], 
               c=colors[class_idx], s=100, alpha=0.6, 
               label=class_names[class_idx], 
               edgecolors='black', linewidths=1)
ax1.set_xlabel('First Principal Component', fontsize=12)
ax1.set_ylabel('Second Principal Component', fontsize=12)
ax1.set_title('True Class Labels', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Cluster assignments on PCA space
for cluster_idx in range(3):
    mask = cluster_labels_3_pca == cluster_idx
    ax2.scatter(PC1[mask], PC2[mask], 
               c=colors[cluster_idx], s=100, alpha=0.6, 
               label=f'Cluster {cluster_idx}', 
               edgecolors='black', linewidths=1)
ax2.set_xlabel('First Principal Component', fontsize=12)
ax2.set_ylabel('Second Principal Component', fontsize=12)
ax2.set_title('K-means on PCA Scores (K=3)', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("COMMENTS:")
print("="*70)
print("Clustering on the first two principal components:")
print("  - Reduces dimensionality from 50 to 2 features")
print("  - Focuses on the directions of maximum variance")
print("  - May perform similarly or even better if the first 2 PCs capture")
print("    most of the class separation information")
print("  - Computationally more efficient (working with 60x2 instead of 60x50)")
print("  - Results depend on how well the first 2 PCs represent the class structure")


## Part (g): Perform K-means Clustering After Scaling (K=3)

**Task:**
- Use `StandardScaler()` to scale each variable to have standard deviation of 1
- Perform K-means clustering with K = 3 on the scaled data
- Compare results to those obtained in part (c)
- Explain any differences or similarities


In [ ]:
# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("="*70)
print("DATA SCALING:")
print("="*70)
print(f"\nOriginal data statistics (first 5 features):")
print(f"  Means: {X[:, :5].mean(axis=0)}")
print(f"  Std devs: {X[:, :5].std(axis=0)}")

print(f"\nScaled data statistics (first 5 features):")
print(f"  Means: {X_scaled[:, :5].mean(axis=0)}")
print(f"  Std devs: {X_scaled[:, :5].std(axis=0)}")

# Perform K-means clustering on scaled data
kmeans3_scaled = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels_3_scaled = kmeans3_scaled.fit_predict(X_scaled)

# Create crosstab
crosstab_3_scaled = pd.crosstab(pd.Series(true_labels, name='True Class'),
                                 pd.Series(cluster_labels_3_scaled, name='Cluster'),
                                 margins=True)

print("\n" + "="*70)
print("K-MEANS CLUSTERING ON SCALED DATA (K=3):")
print("="*70)
print("\nCrosstab: True Class vs Cluster Assignment")
print(crosstab_3_scaled)

# Calculate accuracy
cost_matrix_scaled = np.zeros((3, 3))
for true_class in range(3):
    for cluster in range(3):
        count = np.sum((true_labels == true_class) & (cluster_labels_3_scaled == cluster))
        cost_matrix_scaled[true_class, cluster] = -count

row_ind_scaled, col_ind_scaled = linear_sum_assignment(cost_matrix_scaled)

correct_scaled = 0
for true_class, cluster in zip(row_ind_scaled, col_ind_scaled):
    correct_scaled += np.sum((true_labels == true_class) & (cluster_labels_3_scaled == cluster))

accuracy_3_scaled = correct_scaled / len(true_labels)

print(f"\nOptimal cluster-to-class mapping:")
for true_class, cluster in zip(row_ind_scaled, col_ind_scaled):
    print(f"  True Class {true_class} → Cluster {cluster}")

print(f"\nAccuracy (with optimal mapping): {accuracy_3_scaled:.4f} ({accuracy_3_scaled*100:.2f}%)")

# Perform PCA on scaled data for visualization
pca_scaled = PCA()
X_pca_scaled = pca_scaled.fit_transform(X_scaled)
PC1_scaled = X_pca_scaled[:, 0]
PC2_scaled = X_pca_scaled[:, 1]

variance_explained_scaled = pca_scaled.explained_variance_ratio_
print(f"\nPCA on scaled data:")
print(f"  Variance explained by first PC: {variance_explained_scaled[0]:.4f} ({variance_explained_scaled[0]*100:.2f}%)")
print(f"  Variance explained by second PC: {variance_explained_scaled[1]:.4f} ({variance_explained_scaled[1]*100:.2f}%)")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: True labels (original PCA)
ax = axes[0, 0]
for class_idx in range(n_classes):
    mask = true_labels == class_idx
    ax.scatter(PC1[mask], PC2[mask], 
               c=colors[class_idx], s=100, alpha=0.6, 
               label=class_names[class_idx], 
               edgecolors='black', linewidths=1)
ax.set_xlabel('First Principal Component', fontsize=12)
ax.set_ylabel('Second Principal Component', fontsize=12)
ax.set_title('True Class Labels (Original Data PCA)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Clustering on original data
ax = axes[0, 1]
for cluster_idx in range(3):
    mask = cluster_labels_3 == cluster_idx
    ax.scatter(PC1[mask], PC2[mask], 
               c=colors[cluster_idx], s=100, alpha=0.6, 
               label=f'Cluster {cluster_idx}', 
               edgecolors='black', linewidths=1)
ax.set_xlabel('First Principal Component', fontsize=12)
ax.set_ylabel('Second Principal Component', fontsize=12)
ax.set_title(f'K-means on Original Data (K=3)\nAccuracy: {accuracy_3:.2%}', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: True labels (scaled data PCA)
ax = axes[1, 0]
for class_idx in range(n_classes):
    mask = true_labels == class_idx
    ax.scatter(PC1_scaled[mask], PC2_scaled[mask], 
               c=colors[class_idx], s=100, alpha=0.6, 
               label=class_names[class_idx], 
               edgecolors='black', linewidths=1)
ax.set_xlabel('First Principal Component', fontsize=12)
ax.set_ylabel('Second Principal Component', fontsize=12)
ax.set_title('True Class Labels (Scaled Data PCA)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Clustering on scaled data
ax = axes[1, 1]
for cluster_idx in range(3):
    mask = cluster_labels_3_scaled == cluster_idx
    ax.scatter(PC1_scaled[mask], PC2_scaled[mask], 
               c=colors[cluster_idx], s=100, alpha=0.6, 
               label=f'Cluster {cluster_idx}', 
               edgecolors='black', linewidths=1)
ax.set_xlabel('First Principal Component', fontsize=12)
ax.set_ylabel('Second Principal Component', fontsize=12)
ax.set_title(f'K-means on Scaled Data (K=3)\nAccuracy: {accuracy_3_scaled:.2%}', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("COMPARISON: Original vs Scaled Data")
print("="*70)
print(f"\nClustering on original data:")
print(f"  Accuracy: {accuracy_3:.4f} ({accuracy_3*100:.2f}%)")

print(f"\nClustering on scaled data:")
print(f"  Accuracy: {accuracy_3_scaled:.4f} ({accuracy_3_scaled*100:.2f}%)")

print(f"\nDifference: {abs(accuracy_3 - accuracy_3_scaled):.4f}")

print("\n" + "="*70)
print("EXPLANATION:")
print("="*70)
print("Scaling standardizes each variable to have mean 0 and standard deviation 1.")
print("This is important for K-means because:")
print("  1. K-means uses Euclidean distance, which is sensitive to feature scales")
print("  2. Variables with larger scales dominate the distance calculation")
print("  3. Without scaling, features with larger variance have more influence")
print("\nIn this case:")
print("  - The original data has different means for each class (mean shifts)")
print("  - All features have similar scales (std=1, different means)")
print("  - Scaling may change the relative importance of features")
print("  - Results may differ because the distance metric changes")
print("\nIf features have very different scales, scaling is crucial.")
print("If features have similar scales (as in this simulation), the difference")
print("may be smaller, but scaling is still a good practice.")


## Summary

This lab demonstrated:
1. **Data Simulation**: Generated data with distinct classes using mean shifts
2. **PCA**: Reduced dimensionality and visualized class separation in principal component space
3. **K-means Clustering**: Explored clustering with different K values (2, 3, 4)
4. **PCA-based Clustering**: Applied K-means to reduced-dimensional data
5. **Scaling Effects**: Compared clustering results with and without feature scaling

**Key Takeaways:**
- K-means requires the correct number of clusters (K) to match the true structure
- PCA can help visualize and reduce dimensionality for clustering
- Feature scaling is important when variables have different scales
- Clustering results should be interpreted carefully, especially when K doesn't match the true number of classes
